# பாடம் 10 - ஏ.ஐ. முகவர்கள் உற்பத்தி சூழலில்

இந்த பாடத்தில் நீங்கள் Microsoft Agent Framework (Python) பயன்படுத்தி ஏ.ஐ. முகவர்களுக்கான **உற்பத்தி வடிவ முறைமைகள்** பற்றி கற்றுக்கொள்வீர்கள். நாம் கையாள்பவைகள்:

- **Observability** — முகவர் தொடர்புகளில் நேர அளவீடு மற்றும் பதிவு சேர்க்குதல்
- **Evaluation** — பதில் தரத்தை மதிப்பிட ஒரு மதிப்பீட்டாளர் முகவரைப் பயன்படுத்துதல்
- **Cost management** — டோக்கன் ஒப்டிமைசேஷன் மற்றும் மாதிரி தேர்வுக்கான வியூகங்கள்

காட்சி ஒரு **பயண முகவர்**; பயனர்களுக்கு பயணத் திட்டமிட உதவுகிறது, அதில் கண்காணிப்பு மற்றும் மதிப்பீடு அடுக்குகள் உள்ளன.


## அமைப்பு


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## தயாரிப்புக்கான பரிசீலனைகள்

AI ஏஜென்ட்களை மாதிரிகளிலிருந்து தயாரிப்புக்கு மாற்றுதல் மூன்று முக்கிய தூண்களுக்கு மிகுந்த கவனத்தை தேவைபடுத்துகிறது:

1. **காணக்கூடிய தன்மை** — ஏஜென்ட் என்ன செய்கிறது, அதற்கு எவ்வளவு நேரம் செலவாகிறது, மற்றும் அது எந்த கருவிகளை அழைக்கிறது என்பதை நீங்கள் தெளிவாக காண வேண்டும். பின்தொடர்தலும் பதிவேட்டும் இல்லாவிட்டால், தயாரிப்பில் உள்ள சிக்கல்களை பிழைத்திருத்துவது கிட்டத்தட்ட சாத்தியமில்லை.

2. **மதிப்பீடு** — தானியங்கி தரச் சரிபார்ப்புகள் காலப்போக்கில் ஏஜென்டின் பதில்கள் துல்லியமாகவும், முழுமையாகவும் மற்றும் உதவிகரமாகவும் இருக்குமாறு உறுதிசெய்கின்றன. ஒரு மதிப்பீட்டு ஏஜெண்ட் குறிப்பிட்ட அடிப்படைக் கணக்கீடுகளுக்கு எதிராக பதில்களை மதிப்பெண் பகிர்ந்து மதிப்பீடு செய்யலாம்.

3. **செலவு மேலாண்மை** — டோக்கன் பயன்பாடு நேரடியாக செலவை பாதிக்கிறது. ப்ராம்ட் ஒப்டிமைசேஷன், மாதிரி தேர்வு மற்றும் கேஷிங் போன்ற நெறிமுறைகள் தரத்தை குருதிக்காமல் செலவுகளை கட்டுப்பாட்டில் வைக்க உதவுகின்றன.


## கண்காணிக்கக்கூடிய ஏஜெண்டை கட்டமைத்தல்

நாம் பயண கருவிகளை வரையறுக்கிறோம் மற்றும் தாமதத்தை கண்காணிக்க ஏஜெண்ட் அழைப்பை நேர அளவீட்டுடன் சுற்றிக்கொள்கிறோம். உற்பத்தியில் நீங்கள் OpenTelemetry அல்லது இதே மாதிரியான டிரேசிங் பின்புறத்துடன் ஒருங்கிணைக்கலாம்.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## மதிப்பீட்டு மாதிரிகள்

ஒரு பொதுவான உற்பத்தி மாதிரி என்பது இரண்டாவது ஏஜென்டை **மதிப்பீட்டாளராக** பயன்படுத்துவது. மதிப்பீட்டாளர் முதன்மை ஏஜென்டின் பதிலை முழுமை, துல்லியம் மற்றும் உதவித்தன்மை போன்ற முன்-நிர்ணயிக்கப்பட்ட அளவுகோல்களின் அடிப்படையில் மதிப்பிடுவார்.

இதனால்:
- பதில்கள் பயனர்களுக்கு செல்லுமுன் தானாக இயங்கும் தரக் கட்டுப்பாடுகள்
- ப்ராம்ப்டுகள் அல்லது மாதிரிகள் மாறும்போது செயல்திறன் குறைவுகளை கண்டறிதல்
- நீண்டகாலத்தில் ஏஜென்டின் செயல்திறனை தொடர்ச்சியாக கண்காணித்தல்


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## செலவு மேலாண்மை நெறிமுறைகள்

உற்பத்தி AI代理களின் செலவுகளை கட்டுப்படுத்துவது மிக அவசியம். இங்கே முக்கியமான நெறிமுறைகள் உள்ளன:

| Strategy | Description |
|---|---|
| **ப்ராம்ட் மேம்படுத்தல்** | சிஸ்டம் உத்தரவுகளை சுருக்கமாக வைத்திருங்கள். உள்ளீட்டு டோக்கன்களை குறைக்க அதேபோன்ற சூழ்நிலைகளை நீக்குங்கள். |
| **மாடல் தேர்வு** | எளிய பணிகளுக்கான வகைப்படுத்தல் அல்லது மீட்டெடுப்பு போன்றவற்றிற்கு சிறியது மற்றும் குறைந்த செலவுடைய மாடல்களை (உதா. GPT-4o-mini) பயன்படுத்தவும், அதிக திறன் தேவைப்படும் சிக்கலான யூகத்திற்காக பெரிய மாடல்களை ஒதுக்கி வைக்கவும். |
| **கேஷிங்** | கருவி முடிவுகளையும் அடிக்கடி வரும் வினவுதல்களையும் கேஷ் செய்து மறு API அழைப்புகளைத் தவிர்க்கவும். |
| **டோக்கன் பட்ஜெட்டுகள்** | அதிர்ச்சியாக நீளமான பதில்களைத் தடுக்கும் வகையில் `max_tokens` வரம்புகளை அமைக்கவும். |
| **Batching** | பொருந்தும் இடங்களில் பல பயனர் கேள்விகளை ஒரே API அழைப்பாகத் தொகுத்து அனுப்புங்கள். |

பயன்பாட்டில், அடுக்கு (tiered) அணுகுமுறை நன்றாக வேலைசெய்கிறது: நேரடியாகச் சொல்லப்படுகிற கோரிக்கைகளை வேகமான, மலிவான மாடலுக்கு அனுப்பி, சிக்கலான கேள்விகளை மட்டுமே திறன் வாய்ந்த மற்றொரு மாடலுக்குக் குறுக்குங்கள்.


## சுருக்கம்

இந்த பாடத்தில் நீங்கள் எப்படி செய்வது என்பதை கற்றுக்கொண்டீர்கள்:

1. **எஜெண்ட் தொடர்புகளுக்கு நேர கணக்கெடுப்பு மற்றும் பதிவு மூலம் கண்காணிக்கக்கூடிய தன்மையைச் சேர்க்கவும்**, இதனால் தடம் பின்தொடர்தலும் கண்காணிப்பும் சாத்தியமாகின்றன.
2. **எஜெண்ட் பதில்களை தன்னிச்சையாக மதிப்பிடவும்** — முழுமை, துல்லியம் மற்றும் உதவித்தன்மைக்கு மதிப்பெண்களை வழங்கும் ஒரு மதிப்பாய்வாளர் எஜெண்ட்டை பயன்படுத்தி.
3. **செலவுகளை நிர்வகிக்கவும்** prompt ஒப்டிமைசேஷன், மாடல் தேர்வு, கேச்சிங், மற்றும் டோக்கன் பட்ஜெட்டுகள் மூலம்.

இவை உற்பத்தி மாதிரிகள் உங்கள் AI எஜெண்ட்கள் நம்பகமான, அளவிடக்கூடிய, மற்றும் பெரிய அளவில் செலவுச் சக்கரத்திற்கு ஏற்றவையாக இருக்க உதவுகின்றன.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
மறுப்பு:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவையான [Co-op Translator](https://github.com/Azure/co-op-translator) மூலம் மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சித்தாலும், தானியங்கி மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறான தகவல்கள் இருக்க வாய்ப்புள்ளது என்பதை கவனத்தில் கொள்ளவும். மூல ஆவணம் அதன் சொந்த மொழியில் அதிகாரப்பூர்வ மூலமாகக் கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்முறை மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பின் பயன்பாட்டினாலோ அதன் விளைவுகளினாலோ ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கங்களுக்கும் நாங்கள் பொறுப்பேற்க மாட்டோம்.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
